# Lesson 7: A complete tiny GPT forward pass

Lessons 4–6 built the model inputs and one decoder block. Now we connect them into the same forward-pass sequence as GPT-2: token embeddings plus position embeddings, one decoder block, a final LayerNorm, and vocabulary logits.

Our tiny model has one sequence of three token IDs, embedding width 2, vocabulary size 4, and one block. Its weights are fixed teaching values. The output logits show how the last position scores each possible next token; these scores are not probabilities, and the toy model has not been trained.

**Read the companions:** [C gpt_forward_lesson.c](../../c/src/gpt_forward_lesson.c) and [Java GptForwardLesson.java](../../java/src/main/java/learning/karpathy/gpt2/GptForwardLesson.java). Run them with `-Action lesson -Lesson 7`.


In [ ]:
import math
import sys

import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")


## 1. Look up token and position vectors

The input is a matrix of integer IDs with shape \((B,T)=(1,3)\). The token table maps each ID into \(C=2\) channels. The position table supplies a learned vector for each location. Their sum is the initial residual stream \((B,T,C)=(1,3,2)\).

GPT-2 ties its output projection to the token embedding table. We will use that same weight matrix later to turn each hidden vector into one logit per vocabulary token.


In [ ]:
BATCH_SIZE = 1
SEQUENCE_LENGTH = 3
EMBEDDING_SIZE = 2
VOCAB_SIZE = 4
MLP_WIDTH = 4 * EMBEDDING_SIZE
EPSILON = 1e-5
dtype = torch.float64

token_ids = torch.tensor([[1, 2, 3]], dtype=torch.long)
token_embedding_weight = torch.tensor(
    [[0.10, 0.20], [0.30, -0.10], [-0.20, 0.40], [0.05, 0.25]],
    dtype=dtype,
)
position_embedding_weight = torch.tensor(
    [[0.01, 0.00], [0.00, 0.02], [0.03, -0.01]],
    dtype=dtype,
)

# nn.Embedding is a table lookup: one learned row for each integer ID.
token_vectors = F.embedding(token_ids, token_embedding_weight)
position_ids = torch.arange(SEQUENCE_LENGTH)
position_vectors = F.embedding(position_ids, position_embedding_weight)
x = token_vectors + position_vectors[None, :, :]

print("Token IDs (B, T):\n", token_ids)
print("Token vectors (B, T, C):\n", token_vectors)
print("Position vectors (T, C):\n", position_vectors)
print("Initial residual stream x = token + position (B, T, C):\n", x)


## 2. Run the decoder block

We reuse the one-head, identity-projection attention and the same fixed MLP weights from Lesson 6. In GPT-2 these projections and the MLP weights are learned; here the small constants make the full data flow reproducible in C and Java.

The block keeps the hidden shape \((B,T,C)\) unchanged. Only the MLP's intermediate representation expands to \((B,T,4C)\).


In [ ]:
identity = torch.eye(EMBEDDING_SIZE, dtype=dtype)
gamma_1 = torch.ones(EMBEDDING_SIZE, dtype=dtype)
beta_1 = torch.zeros(EMBEDDING_SIZE, dtype=dtype)
gamma_2 = torch.ones(EMBEDDING_SIZE, dtype=dtype)
beta_2 = torch.zeros(EMBEDDING_SIZE, dtype=dtype)

W_MLP_IN = torch.tensor(
    [[0.20, -0.10], [-0.10, 0.25], [0.30, 0.05], [0.00, 0.20],
     [-0.20, 0.10], [0.10, -0.15], [0.15, 0.00], [-0.05, 0.30]], dtype=dtype,
)
W_MLP_OUT = torch.tensor(
    [[0.10, -0.10, 0.20, 0.00, 0.10, -0.15, 0.05, 0.10],
     [-0.20, 0.15, 0.05, 0.20, 0.10, 0.05, -0.05, 0.20]], dtype=dtype,
)

x_norm_1 = F.layer_norm(x, (EMBEDDING_SIZE,), gamma_1, beta_1, EPSILON)
query = F.linear(x_norm_1, identity)
key = F.linear(x_norm_1, identity)
value = F.linear(x_norm_1, identity)
attention_context = F.scaled_dot_product_attention(
    query[:, None, :, :], key[:, None, :, :], value[:, None, :, :], is_causal=True,
)[:, 0, :, :]
after_attention = x + F.linear(attention_context, identity)

x_norm_2 = F.layer_norm(after_attention, (EMBEDDING_SIZE,), gamma_2, beta_2, EPSILON)
mlp_pre_activation = F.linear(x_norm_2, W_MLP_IN)
mlp_activated = F.gelu(mlp_pre_activation, approximate="tanh")
mlp_branch = F.linear(mlp_activated, W_MLP_OUT)
block_output = after_attention + mlp_branch

print("After attention residual (B, T, C):\n", after_attention)
print("MLP hidden state (B, T, 4C):\n", mlp_activated)
print("Decoder block output (B, T, C):\n", block_output)


## 3. Normalize and produce vocabulary logits

GPT-2 applies a final LayerNorm, then a linear projection from \(C\) channels to the vocabulary size. Weight tying means the projection reuses the token embedding table, transposed for the linear operation.

The result has shape \((B,T,V)=(1,3,4)\). For next-token generation, inspect the logits at the final sequence position and choose the largest score. A softmax could convert these into probabilities, but the model has no learned language knowledge because these are toy weights.


In [ ]:
final_gamma = torch.ones(EMBEDDING_SIZE, dtype=dtype)
final_beta = torch.zeros(EMBEDDING_SIZE, dtype=dtype)
normalized = F.layer_norm(block_output, (EMBEDDING_SIZE,), final_gamma, final_beta, EPSILON)

# The same tensor is used as the embedding table and output projection weight.
lm_head_weight = token_embedding_weight
logits = F.linear(normalized, lm_head_weight)
next_token_logits = logits[0, -1]
predicted_token_id = int(torch.argmax(next_token_logits))

print("Final normalized hidden states (B, T, C):\n", normalized)
print("All vocabulary logits (B, T, V):\n", logits)
print("Next-token logits at the final position:", next_token_logits)
print(f"Toy next-token choice: token ID {predicted_token_id}")


## Check the end-to-end shapes and connections

The dimensions follow through the model without changing the sequence length. We also check that the output projection really shares the embedding weights and that all logits are finite.

**Connect to the reference:** `GPT.forward` in Karpathy's source performs these same stages: `wte(idx) + wpe(pos)`, each block, `ln_f`, then `lm_head`. Our forward pass has one block and toy tables rather than GPT-2's 12 blocks and trained parameters.

**Try it:** change the input IDs while keeping positions fixed, then change only one position vector. Compare the last-position logits. You can also break weight tying by cloning `token_embedding_weight` before using it in `F.linear` and observe the separate parameter table.

**References:** [reference GPT.forward](../../build-nanogpt/train_gpt2.py#L91), [PyTorch Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html), [PyTorch LayerNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html), and [PyTorch Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html).


In [ ]:
assert tuple(token_ids.shape) == (BATCH_SIZE, SEQUENCE_LENGTH)
assert tuple(token_vectors.shape) == (BATCH_SIZE, SEQUENCE_LENGTH, EMBEDDING_SIZE)
assert tuple(block_output.shape) == (BATCH_SIZE, SEQUENCE_LENGTH, EMBEDDING_SIZE)
assert tuple(logits.shape) == (BATCH_SIZE, SEQUENCE_LENGTH, VOCAB_SIZE)
assert lm_head_weight.data_ptr() == token_embedding_weight.data_ptr()
assert torch.isfinite(logits).all()
assert predicted_token_id == int(torch.argmax(logits[0, -1]))
print("Embedding, block, and logits shapes are correct: yes")
print("Output projection shares the token embedding weights: yes")
print("All logits are finite; final-position argmax is a valid toy next-token choice: yes")
